# PhasePhyto -- Batch Inference Across One or More Datasets

Run one or more trained PhasePhyto checkpoint suites over every image in one or
more target datasets (PlantDoc, Cassava, PlantVillage-style roots, or a custom
image folder). Writes per-image, per-model predictions + confidence to CSV,
plus cross-model agreement/disagreement reports for each dataset and one master
summary across datasets.

**Why this exists.** Our reported ablation numbers (see `RESULTS.md`,
`PAPER.md`) are on the 25-class mapped PlantDoc subset (n=153). This notebook
widens the lens:

1. **Multi-dataset sweep.** Pass a nested `DATASET_RUNS` dict and evaluate the
   same four trained variants across PlantDoc, Cassava, or custom datasets in
   one Colab session.
2. **Dataset preflight checks.** Before inference starts, the notebook verifies
   that every referenced dataset root exists, resolves nested split layouts
   (e.g. `plantdoc/test`), and checks that the required checkpoint variants are
   present.
3. **Cross-model agreement.** Running all four ablation checkpoints
   (`full`, `backbone_only`, `no_fusion`, `pc_only`) on every image gives a
   per-image "models agree" / "models disagree" flag. Directly attacks the
   `full` vs `backbone_only` noise-vs-signal question in `PAPER.md`
   Limitations.
4. **Confidence distributions.** Per-class max-softmax histograms under each
   ablation let you calibrate the pseudo-label threshold against the full target
   set, not just the 153-image mapped subset.

## How to run

### A) One-time setup

1. In Colab, mount Drive (cell 2) so checkpoints and outputs persist.
2. The notebook clones `PhasePhyto` from GitHub; if you maintain a fork,
   edit `REPO_URL` / `REPO_BRANCH` in cell 3.
3. Ensure your target datasets already exist in Drive or local Colab storage.
   If you used `PhasePhyto_Download_Data_To_Drive.ipynb`, you can point this
   notebook at its `dataset_manifest.json` and omit individual dataset roots.

### B) Per-run configuration (cell 5 -- the only cell you normally edit)

Fill in `DATASET_RUNS` with one entry per dataset sweep. Each entry accepts:

- `dataset_kind`: one of `plantdoc`, `cassava`, `plantvillage`, `plant_pathology_2021`, `rocole`, `rice_leaf`, `banana_leaf`, `custom`
- `dataset_root`: optional if discoverable from `dataset_manifest.json`
- `class_to_idx_source`: optional if the first checkpoint embeds `class_to_idx`
- `checkpoints`: dict of the four ablations -> checkpoint path or `{path, name}`

The notebook will fail fast if a dataset is missing or if one of the four
required ablations is absent.


## 0. Environment & Dependencies


In [ ]:
# Install extra dependencies not already present in Colab.
!pip install -q timm opencv-python scikit-learn matplotlib seaborn tqdm


In [ ]:
# Mount Google Drive (skip if running locally / checkpoint paths are already
# local). Harmless to re-run if already mounted.
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Colab; skipping Drive mount.")


In [ ]:
# Clone / update the PhasePhyto repo so we can reuse its model components.
# Edit REPO_URL / REPO_BRANCH if you maintain a fork.
import os
import subprocess

REPO_URL = "https://github.com/kautilyaa/PhasePhyto.git"
REPO_BRANCH = "aws_changes"
REPO_DIR = "/content/PhasePhyto"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--quiet"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH, "--quiet"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--quiet"], check=False)

# Editable install so `from phasephyto.models import ...` works downstream.
subprocess.run(["pip", "install", "-q", "-e", REPO_DIR], check=True)

# Make repo importable without reinstall in this session.
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready at:", REPO_DIR)


## 1. Configuration -- EDIT THIS CELL

The only cell you normally need to edit. Fill in the checkpoint registry
and output paths, then run top-to-bottom.


In [ ]:
# =============================================================================
# BATCH INFERENCE CONFIG -- edit freely before running
# =============================================================================

# Optional shared dataset manifest created by PhasePhyto_Download_Data_To_Drive.
# If present, dataset roots can be omitted per run and will be resolved from it.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/PhasePhyto"
DATASET_MANIFEST_PATH = f"{DRIVE_PROJECT_DIR}/data/plant_disease/dataset_manifest.json"

# Default source label-space reference. For PlantVillage->PlantDoc this should
# point to the PlantVillage root (or a class_to_idx json). If omitted inside a
# dataset spec, the notebook falls back to this value, and then finally to the
# first checkpoint path (if that checkpoint embeds `class_to_idx`).
DEFAULT_CLASS_TO_IDX_SOURCE = "/content/data/plantvillage"

# Enforce the full four-ablation suite by default.
REQUIRE_ALL_VARIANTS = True

# --- Dataset runs -----------------------------------------------------------
# Each run config supports:
#   dataset_kind:        one of {"plantdoc", "cassava", "plantvillage", "plant_pathology_2021", "rocole", "rice_leaf", "banana_leaf", "custom"}
#   dataset_root:        optional if DATASET_MANIFEST_PATH provides it
#   class_to_idx_source: optional override for the source label space
#   checkpoints:         dict keyed by ablation name; values can be either:
#                        - "/abs/path/to/checkpoint.pt"
#                        - {"path": "/abs/path/to/checkpoint.pt", "name": "pretty_label"}
#
# Example structure (replace placeholders with your real paths):
DATASET_RUNS = {
    # "plantdoc_all": {
    #     "dataset_kind": "plantdoc",
    #     # "dataset_root": "/content/data/plantdoc",  # optional if manifest exists
    #     "class_to_idx_source": "/content/data/plantvillage",
    #     "checkpoints": {
    #         "full": {"path": "/content/drive/MyDrive/PhasePhyto/runs/.../best_phasephyto.pt", "name": "full_leafmask"},
    #         "backbone_only": "/content/drive/MyDrive/PhasePhyto/runs/.../backbone_only.pt",
    #         "no_fusion": "/content/drive/MyDrive/PhasePhyto/runs/.../no_fusion.pt",
    #         "pc_only": "/content/drive/MyDrive/PhasePhyto/runs/.../pc_only.pt",
    #     },
    # },
    # "cassava_holdout": {
    #     "dataset_kind": "cassava",
    #     # "dataset_root": "/content/data/cassava",  # optional if manifest exists
    #     "checkpoints": {
    #         "full": "/content/drive/MyDrive/PhasePhyto/runs/.../cassava_full.pt",
    #         "backbone_only": "/content/drive/MyDrive/PhasePhyto/runs/.../cassava_backbone_only.pt",
    #         "no_fusion": "/content/drive/MyDrive/PhasePhyto/runs/.../cassava_no_fusion.pt",
    #         "pc_only": "/content/drive/MyDrive/PhasePhyto/runs/.../cassava_pc_only.pt",
    #     },
    # },
}

# --- Where to save outputs --------------------------------------------------
from datetime import datetime
_run_tag = datetime.now().strftime("%Y%m%d-%H%M%S")
OUTPUT_DIR = f"/content/drive/MyDrive/PhasePhyto/batch_inference/{_run_tag}"

# --- Inference knobs --------------------------------------------------------
BATCH_SIZE = 32          # lower if OOM on PC stream (PC extractor is memory-heavy)
NUM_WORKERS = 2          # DataLoader workers
IMAGE_SIZE = 224         # must match training (checkpoints use 224)
USE_LEAF_MASK_FROM_CONFIG = True   # if True, each checkpoint's embedded
                                   # config drives leaf_mask_mode for its
                                   # own val transform. If False, LEAF_MASK_OVERRIDE
                                   # below is used for every checkpoint.
LEAF_MASK_OVERRIDE = "off"         # ignored if USE_LEAF_MASK_FROM_CONFIG is True
USE_HFLIP_TTA = True               # softmax-averaged over image + hflip
DEVICE_PREF = "cuda"               # "cuda" or "cpu" -- falls back to cpu if no GPU

print(f"Output dir will be: {OUTPUT_DIR}")
print(f"Configured dataset runs: {len(DATASET_RUNS)}")


## 2. Imports & Helpers


In [ ]:
import json
import math
import os
from dataclasses import asdict
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

# Repo modules
from phasephyto.batch_inference_config import (
    CANONICAL_ABLATIONS,
    normalize_dataset_runs,
)
from phasephyto.models.phase_congruency import (
    LogGaborFilterBank,
    PhaseCongruencyExtractor,
)
from phasephyto.models.pc_encoder import PCEncoder
from phasephyto.models.semantic_backbone import SemanticBackbone
from phasephyto.models.illumination_norm import IlluminationNormStream
from phasephyto.models.cross_attention import StructuralSemanticFusion
from phasephyto.data.class_mapping import (
    PLANTDOC_TO_PLANTVILLAGE,
    normalize_class_name,
)

DEVICE = torch.device(
    "cuda" if (DEVICE_PREF == "cuda" and torch.cuda.is_available()) else "cpu"
)
print("Device:", DEVICE)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Output dir created:", OUTPUT_DIR)


## 3. Ablation-Aware PhasePhyto Model

Mirrors the definition in the training notebook (cell 34). The only reason
we inline this is that the packaged `phasephyto.models.PhasePhyto` does not
take an `ablation` flag; the training notebook is the source of truth for
the four-way branch.


In [ ]:
_ABLATIONS = ("full", "pc_only", "backbone_only", "no_fusion")


class PhasePhyto(nn.Module):
    """Ablation-aware PhasePhyto, compatible with training-notebook checkpoints."""

    def __init__(
        self,
        num_classes=25,
        backbone_name="vit_base_patch16_224",
        fusion_dim=256,
        pc_scales=4,
        pc_orientations=6,
        image_size=(224, 224),
        num_heads=4,
        dropout=0.1,
        pretrained=False,
        freeze_backbone=False,
        ablation="full",
    ):
        super().__init__()
        if ablation not in _ABLATIONS:
            raise ValueError(f"ablation must be one of {_ABLATIONS}, got {ablation!r}")
        self.ablation = ablation
        self.image_size = image_size

        self.filter_bank = LogGaborFilterBank(image_size, pc_scales, pc_orientations)
        self.pc_extractor = PhaseCongruencyExtractor(pc_scales, pc_orientations)
        self.pc_encoder = PCEncoder(in_channels=3, fusion_dim=fusion_dim)

        self.backbone = SemanticBackbone(backbone_name, fusion_dim, pretrained, freeze_backbone)
        self.illum_stream = IlluminationNormStream(out_dim=fusion_dim)
        self.fusion = StructuralSemanticFusion(fusion_dim, num_heads, dropout)

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes),
        )
        self.aux_pc_head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes),
        )

        self.register_buffer(
            "lum_weights",
            torch.tensor([0.2989, 0.5870, 0.1140]).view(1, 3, 1, 1),
        )

    def forward(self, x_rgb, x_clahe=None, ablation=None):
        abl = ablation or self.ablation
        if x_clahe is None:
            x_clahe = x_rgb

        compute_pc = abl in ("full", "pc_only", "no_fusion")
        compute_backbone = abl in ("full", "backbone_only", "no_fusion")

        structural_tokens = None
        if compute_pc:
            gray = (x_rgb * self.lum_weights).sum(dim=1, keepdim=True)
            even, odd = self.filter_bank(gray)
            pc = self.pc_extractor(even, odd)
            pc_in = torch.cat(
                [pc["pc_magnitude"], pc["phase_symmetry"], pc["oriented_energy"]], dim=1
            )
            structural_tokens = self.pc_encoder(pc_in)

        semantic_tokens = None
        if compute_backbone:
            semantic_tokens = self.backbone(x_rgb)

        illum = self.illum_stream(x_clahe)

        if abl == "full":
            fused, _ = self.fusion(structural_tokens, semantic_tokens, False)
        elif abl == "pc_only":
            fused = structural_tokens.mean(dim=1)
        elif abl == "backbone_only":
            fused = semantic_tokens.mean(dim=1)
        elif abl == "no_fusion":
            fused = 0.5 * (structural_tokens.mean(dim=1) + semantic_tokens.mean(dim=1))
        else:
            raise ValueError(f"Unknown ablation: {abl!r}")

        return self.classifier(torch.cat([fused, illum], dim=1))


## 4. Transforms (validation only)

Mirrors the training notebook's `val_tf` pipeline but without any
augmentation. Produces `(rgb_tensor, clahe_tensor)` per image.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img_np):
        if not isinstance(img_np, np.ndarray):
            img_np = np.array(img_np)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = self.clahe.apply(l)
        return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def _hsv_leaf_mask(img_np, sat_thresh=40):
    hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)
    return (hsv[..., 1] >= sat_thresh).astype(np.uint8)


def _apply_leaf_mask(img_np, mode, sat_thresh=40, blur_sigma=1.5):
    if mode == "off":
        return img_np
    mask = _hsv_leaf_mask(img_np, sat_thresh=sat_thresh)
    if mask.sum() < 100:
        return img_np
    fg = img_np[mask.astype(bool)]
    if mode == "hsv":
        mean_c = fg.reshape(-1, 3).mean(axis=0).astype(np.uint8)
        bg_fill = np.ones_like(img_np) * mean_c
    elif mode == "hsv_blur":
        k = max(3, int(blur_sigma * 6) | 1)
        bg_fill = cv2.GaussianBlur(img_np, (k, k), blur_sigma)
    else:
        return img_np
    m = mask[..., None].astype(np.float32)
    return (img_np.astype(np.float32) * m + bg_fill.astype(np.float32) * (1 - m)).astype(np.uint8)


class ValDualTransform:
    """Validation DualTransform: deterministic, returns (rgb_tensor, clahe_tensor)."""
    def __init__(self, image_size=224, leaf_mask_mode="off",
                 leaf_mask_sat_thresh=40, leaf_mask_blur_sigma=1.5):
        self.base = transforms.Compose([
            transforms.Resize((image_size, image_size)),
        ])
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        self.to_tensor = transforms.ToTensor()
        self.clahe_fn = CLAHETransform()
        self.leaf_mask_mode = leaf_mask_mode
        self.leaf_mask_sat_thresh = leaf_mask_sat_thresh
        self.leaf_mask_blur_sigma = leaf_mask_blur_sigma

    def __call__(self, image):
        img = self.base(image)
        img_np = np.array(img)
        if self.leaf_mask_mode != "off":
            img_np = _apply_leaf_mask(
                img_np, mode=self.leaf_mask_mode,
                sat_thresh=self.leaf_mask_sat_thresh,
                blur_sigma=self.leaf_mask_blur_sigma,
            )
        clahe_np = self.clahe_fn(img_np)
        return (
            self.normalize(self.to_tensor(img_np)),
            self.normalize(self.to_tensor(clahe_np)),
        )


## 5. Dataset: Enumerate All Target Images


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def _discover_images(root: Path) -> list[tuple[Path, str | None]]:
    """Return list of (image_path, folder_name_or_None)."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {root}")

    # Resolve common PlantDoc layouts: root may itself be `test/`, or may
    # contain `test/` / `val/`. Prefer a test split if one is present.
    candidates = [root, root / "test", root / "Test", root / "val", root / "valid"]
    scan_root = next((c for c in candidates if c.exists() and c.is_dir()), root)

    items: list[tuple[Path, str | None]] = []
    for child in sorted(scan_root.iterdir()):
        if child.is_dir():
            for img in sorted(child.rglob("*")):
                if img.suffix.lower() in IMAGE_EXTENSIONS:
                    items.append((img, child.name))
        elif child.suffix.lower() in IMAGE_EXTENSIONS:
            items.append((child, None))
    return items


class BatchInferenceDataset(Dataset):
    """Returns (rgb_tensor, clahe_tensor, image_path_str, folder_name_or_empty)"""
    def __init__(self, items: list[tuple[Path, str | None]], transform: ValDualTransform):
        self.items = items
        self.transform = transform
        self._skipped: list[tuple[str, str]] = []

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, folder = self.items[idx]
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
            rgb, clahe = self.transform(img)
        except (OSError, UnidentifiedImageError) as exc:
            self._skipped.append((str(path), str(exc)))
            # Return a zero tensor with a sentinel folder marker; caller will drop these.
            rgb = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
            clahe = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
            return rgb, clahe, str(path), (folder or ""), 1  # last: is_bad flag
        return rgb, clahe, str(path), (folder or ""), 0


def _build_class_to_idx(source_path: str) -> dict[str, int]:
    """Load class_to_idx from a json dump OR reconstruct from a PV-style root."""
    p = Path(source_path)
    if p.suffix == ".json" and p.is_file():
        with open(p) as f:
            return {k: int(v) for k, v in json.load(f).items()}
    if p.is_dir():
        class_dirs = sorted([d for d in p.iterdir() if d.is_dir()])
        return {d.name: i for i, d in enumerate(class_dirs)}
    # Checkpoint path -- try to read an embedded class_to_idx.
    if p.is_file() and p.suffix == ".pt":
        blob = torch.load(p, map_location="cpu", weights_only=False)
        if "class_to_idx" in blob:
            return {k: int(v) for k, v in blob["class_to_idx"].items()}
    raise ValueError(
        f"Could not resolve class_to_idx from {source_path!r}. "
        "Provide a class_to_idx.json file, a PlantVillage train dir, or a "
        ".pt checkpoint with 'class_to_idx' embedded."
    )


def _map_folder_to_source_class(folder: str | None, source_classes: list[str]) -> str | None:
    if folder is None:
        return None
    # Direct hit first (Cassava / PlantVillage roots have exact names).
    if folder in source_classes:
        return folder
    # PlantDoc -> PlantVillage alias.
    mapped = PLANTDOC_TO_PLANTVILLAGE.get(folder)
    if mapped and mapped in source_classes:
        return mapped
    # Normalized fallback (case / underscore / punctuation insensitive).
    norm_lookup = {normalize_class_name(c): c for c in source_classes}
    cand = norm_lookup.get(normalize_class_name(folder))
    if cand:
        return cand
    return None


In [ ]:
if not DATASET_RUNS:
    raise RuntimeError("DATASET_RUNS is empty -- edit cell 5 to add at least one dataset run before continuing.")

RUN_SPECS = normalize_dataset_runs(
    DATASET_RUNS,
    manifest_path=DATASET_MANIFEST_PATH,
    drive_project_dir=DRIVE_PROJECT_DIR,
    default_class_to_idx_source=DEFAULT_CLASS_TO_IDX_SOURCE,
    require_all_variants=REQUIRE_ALL_VARIANTS,
)

prefight_rows = []
for spec in RUN_SPECS:
    preflight_rows.append({
        "run_name": spec.run_name,
        "dataset_kind": spec.dataset_kind,
        "dataset_root": spec.dataset_root,
        "resolved_root": spec.inspection.resolved_root,
        "num_images": spec.inspection.num_images,
        "num_classes": spec.inspection.num_classes,
        "class_to_idx_source": spec.class_to_idx_source,
        "variants": ", ".join(ck.ablation for ck in spec.checkpoints),
    })

PREFLIGHT_DF = pd.DataFrame(prefight_rows)
PREFLIGHT_DF


## 6. Per-Checkpoint Inference Loop

For each entry in `MODEL_CHECKPOINTS`: load weights, rebuild transforms
based on the checkpoint's embedded config (if present), run inference on
the full target set, and save a CSV.


In [ ]:
def _load_checkpoint(path: str) -> dict[str, Any]:
    blob = torch.load(path, map_location=DEVICE, weights_only=False)
    if "model_state_dict" not in blob:
        raise ValueError(f"{path} does not contain 'model_state_dict'")
    return blob


def _val_tf_for_checkpoint(ckpt: dict) -> ValDualTransform:
    cfg = ckpt.get("config", {}) if USE_LEAF_MASK_FROM_CONFIG else {}
    mode = cfg.get("leaf_mask_mode", LEAF_MASK_OVERRIDE) if USE_LEAF_MASK_FROM_CONFIG else LEAF_MASK_OVERRIDE
    return ValDualTransform(
        image_size=IMAGE_SIZE,
        leaf_mask_mode=mode or "off",
        leaf_mask_sat_thresh=cfg.get("leaf_mask_sat_thresh", 40),
        leaf_mask_blur_sigma=cfg.get("leaf_mask_blur_sigma", 1.5),
    )


def _hflip_batch(x: torch.Tensor) -> torch.Tensor:
    return torch.flip(x, dims=[-1])


@torch.no_grad()
def _run_inference(
    model: PhasePhyto,
    loader: DataLoader,
    ablation: str,
    tta: bool,
    idx_to_class: dict[int, str],
) -> dict[str, list]:
    model.eval()
    records: dict[str, list] = {
        "image_path": [], "folder": [], "is_bad": [],
        "top1_idx": [], "top1_class": [], "top1_conf": [],
        "top5_classes": [], "top5_confs": [],
        "all_confs": [],
    }
    for rgb, clahe, paths, folders, is_bad in tqdm(loader, desc=f"Inference[{ablation}]"):
        rgb = rgb.to(DEVICE, non_blocking=True)
        clahe = clahe.to(DEVICE, non_blocking=True)
        logits = model(rgb, clahe, ablation=ablation)
        probs = F.softmax(logits, dim=-1)
        if tta:
            logits_f = model(_hflip_batch(rgb), _hflip_batch(clahe), ablation=ablation)
            probs = 0.5 * (probs + F.softmax(logits_f, dim=-1))

        top5_vals, top5_idx = probs.topk(min(5, probs.shape[-1]), dim=-1)
        top1_idx = top5_idx[:, 0]
        top1_conf = top5_vals[:, 0]

        for i in range(probs.shape[0]):
            records["image_path"].append(paths[i])
            records["folder"].append(folders[i])
            records["is_bad"].append(int(is_bad[i].item()))
            records["top1_idx"].append(int(top1_idx[i].item()))
            records["top1_class"].append(idx_to_class[int(top1_idx[i].item())])
            records["top1_conf"].append(float(top1_conf[i].item()))
            records["top5_classes"].append(
                [idx_to_class[int(j.item())] for j in top5_idx[i]]
            )
            records["top5_confs"].append([float(v.item()) for v in top5_vals[i]])
            records["all_confs"].append(probs[i].detach().cpu().numpy().tolist())
    return records


def _save_records(records: dict[str, list], out_csv: Path, name: str) -> pd.DataFrame:
    df = pd.DataFrame({
        "image_path": records["image_path"],
        "folder": records["folder"],
        "is_bad": records["is_bad"],
        f"{name}_top1_class": records["top1_class"],
        f"{name}_top1_idx": records["top1_idx"],
        f"{name}_top1_conf": records["top1_conf"],
        f"{name}_top5_classes": [
            "|".join(xs) for xs in records["top5_classes"]
        ],
        f"{name}_top5_confs": [
            "|".join(f"{v:.4f}" for v in xs) for xs in records["top5_confs"]
        ],
    })
    df = df[df["is_bad"] == 0].drop(columns=["is_bad"]).reset_index(drop=True)
    df.to_csv(out_csv, index=False)
    return df


def _slugify(name: str) -> str:
    return "".join(ch.lower() if ch.isalnum() else "_" for ch in name).strip("_")


def _merge_per_model_frames(per_model_dfs: dict[str, pd.DataFrame]) -> pd.DataFrame:
    combined = None
    for name, df in per_model_dfs.items():
        keep_cols = [
            "image_path",
            "folder",
            f"{name}_top1_class",
            f"{name}_top1_idx",
            f"{name}_top1_conf",
        ]
        sub = df[keep_cols]
        combined = sub if combined is None else combined.merge(sub, on=["image_path", "folder"], how="outer")
    if combined is None:
        raise RuntimeError("No per-model predictions available.")
    return combined


def _agreement_matrix(combined: pd.DataFrame, model_names: list[str]) -> pd.DataFrame:
    agreement = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
    for a in model_names:
        for b in model_names:
            if a == b:
                agreement.loc[a, b] = 1.0
                continue
            col_a = f"{a}_top1_class"
            col_b = f"{b}_top1_class"
            both = combined[[col_a, col_b]].dropna()
            if len(both) == 0:
                agreement.loc[a, b] = float("nan")
            else:
                agreement.loc[a, b] = float((both[col_a] == both[col_b]).mean())
    return agreement


def _write_confidence_plot(per_model_dfs: dict[str, pd.DataFrame], out_dir: Path) -> str:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(
        len(per_model_dfs),
        1,
        figsize=(10, 3 * max(1, len(per_model_dfs))),
        sharex=True,
        squeeze=False,
    )
    for i, name in enumerate(per_model_dfs):
        df = per_model_dfs[name]
        ax = axes[i][0]
        ax.hist(df[f"{name}_top1_conf"].values, bins=40, range=(0.0, 1.0), alpha=0.8)
        ax.set_title(f"{name} -- top-1 confidence distribution (n={len(df)})")
        ax.set_xlabel("max softmax")
        ax.set_ylabel("count")
        for q in (0.5, 0.7, 0.9):
            ax.axvline(q, color="red", linestyle=":", alpha=0.5)
    plt.tight_layout()
    conf_plot = out_dir / "per_class_confidence_hist.png"
    plt.savefig(conf_plot, dpi=110, bbox_inches="tight")
    plt.close(fig)
    return str(conf_plot)


def _write_entropy_plot(combined: pd.DataFrame, model_names: list[str], out_dir: Path) -> tuple[str | None, pd.DataFrame]:
    if len(model_names) < 2:
        return None, combined

    import matplotlib.pyplot as plt

    pred_cols = [f"{n}_top1_class" for n in model_names]

    def _entropy(row):
        labels = row.dropna().tolist()
        if not labels:
            return 0.0
        counts = pd.Series(labels).value_counts(normalize=True).values
        return float(-(counts * np.log(counts + 1e-12)).sum())

    combined = combined.copy()
    combined["disagreement_entropy"] = combined[pred_cols].apply(_entropy, axis=1)

    fig = plt.figure(figsize=(8, 4))
    plt.hist(combined["disagreement_entropy"].values, bins=30, alpha=0.85)
    plt.title("Per-image cross-model disagreement entropy")
    plt.xlabel("entropy across model top-1 votes (nats)")
    plt.ylabel("image count")
    dis_plot = out_dir / "disagreement_entropy.png"
    plt.tight_layout()
    plt.savefig(dis_plot, dpi=110, bbox_inches="tight")
    plt.close(fig)
    return str(dis_plot), combined


def _write_run_summary(
    spec,
    out_dir: Path,
    combined: pd.DataFrame,
    model_meta: dict[str, dict],
    agreement_csv: str,
    conf_plot: str,
    dis_plot: str | None,
) -> str:
    mapped_mask = combined["mapped_source_class"].notna()
    summary_lines = [
        "# Batch Inference Run Summary",
        "",
        f"- Run name: `{spec.run_name}`",
        f"- Dataset kind: `{spec.dataset_kind}`",
        f"- Dataset root: `{spec.dataset_root}`",
        f"- Resolved root: `{spec.inspection.resolved_root}`",
        f"- Total images scanned: {len(combined)}",
        f"- Mapped to source label space: {int(mapped_mask.sum())}",
        f"- Unmapped (unknown-class): {int((~mapped_mask).sum())}",
        f"- Checkpoints evaluated: {len(spec.checkpoints)}",
        "",
        "## Per-model accuracy on the mapped subset",
        "",
        "| Model | Ablation | Mapped accuracy | Mean top-1 conf (all) |",
        "|---|---|---:|---:|",
    ]
    for name, meta in model_meta.items():
        correct_col = f"{name}_correct"
        conf_col = f"{name}_top1_conf"
        mapped_acc = float(combined.loc[mapped_mask, correct_col].mean()) if correct_col in combined else float("nan")
        mean_conf = float(combined[conf_col].mean()) if conf_col in combined else float("nan")
        summary_lines.append(
            f"| {name} | {meta['ablation']} | {mapped_acc:.4f} | {mean_conf:.4f} |"
        )

    summary_lines += [
        "",
        "## Cross-model agreement",
        "",
        f"See `{Path(agreement_csv).name}`. Diagonal = 1.0; off-diagonal = fraction",
        "of scanned images where two models' top-1 predictions match.",
        "",
        "## Artifacts",
        "",
        "- `all_models_predictions.csv` -- one row per image, one column set per model.",
        "- `per_model/<name>_predictions.csv` -- raw per-checkpoint output.",
        f"- `{Path(agreement_csv).name}` -- pairwise agreement matrix.",
        f"- `{Path(conf_plot).name}` -- max-softmax histograms.",
    ]
    if dis_plot:
        summary_lines.append(f"- `{Path(dis_plot).name}` -- per-image entropy across models.")
    summary_lines += [
        "",
        "## Next steps",
        "",
        "1. Inspect the disagreement plot or entropy column: images with high entropy are where",
        "   the four ablations diverge. These are the most informative cases for",
        "   any future labeling pass.",
        "2. Inspect per-model confidence histograms for pseudo-label threshold",
        "   calibration on the full target set.",
    ]
    summary_md = out_dir / "RUN_SUMMARY.md"
    summary_md.write_text("\n".join(summary_lines))
    return str(summary_md)


def _process_dataset_run(spec) -> dict[str, Any]:
    dataset_out = Path(OUTPUT_DIR) / _slugify(spec.run_name)
    dataset_out.mkdir(parents=True, exist_ok=True)
    (dataset_out / "per_model").mkdir(parents=True, exist_ok=True)

    class_to_idx = _build_class_to_idx(spec.class_to_idx_source)
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    source_classes = list(class_to_idx.keys())

    items = _discover_images(Path(spec.dataset_root))
    print(f"\n=== Dataset run: {spec.run_name} ===")
    print(f"Dataset kind: {spec.dataset_kind}")
    print(f"Dataset root: {spec.dataset_root}")
    print(f"Resolved root: {spec.inspection.resolved_root}")
    print(f"Discovered {len(items)} images")

    folder_map = {}
    for _, folder in items:
        if folder and folder not in folder_map:
            folder_map[folder] = _map_folder_to_source_class(folder, source_classes)

    per_model_dfs: dict[str, pd.DataFrame] = {}
    model_meta: dict[str, dict] = {}

    for entry in spec.checkpoints:
        name = entry.name
        ablation = entry.ablation
        ckpt_path = entry.path
        print(f"\n--- {name} ({ablation}) ---\nCheckpoint: {ckpt_path}")

        ckpt = _load_checkpoint(ckpt_path)
        cfg = ckpt.get("config", {})

        model = PhasePhyto(
            num_classes=len(class_to_idx),
            backbone_name=cfg.get("backbone_name", "vit_base_patch16_224"),
            fusion_dim=cfg.get("fusion_dim", 256),
            pc_scales=cfg.get("pc_scales", 4),
            pc_orientations=cfg.get("pc_orientations", 6),
            image_size=(IMAGE_SIZE, IMAGE_SIZE),
            num_heads=cfg.get("num_heads", 4),
            dropout=cfg.get("dropout", 0.1),
            pretrained=False,
            ablation=ablation,
        ).to(DEVICE)

        missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
        if missing:
            print(f"  WARN missing keys: {missing[:5]}{' ...' if len(missing) > 5 else ''}")
        if unexpected:
            print(f"  WARN unexpected keys: {unexpected[:5]}{' ...' if len(unexpected) > 5 else ''}")

        val_tf = _val_tf_for_checkpoint(ckpt)
        dataset = BatchInferenceDataset(items, val_tf)
        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
            shuffle=False,
        )

        records = _run_inference(model, loader, ablation, USE_HFLIP_TTA, idx_to_class)
        out_csv = dataset_out / "per_model" / f"{name}_predictions.csv"
        df = _save_records(records, out_csv, name)

        per_model_dfs[name] = df
        model_meta[name] = {
            "ablation": ablation,
            "checkpoint": ckpt_path,
            "epoch": ckpt.get("epoch"),
            "val_f1": ckpt.get("val_f1"),
            "leaf_mask_mode": cfg.get("leaf_mask_mode", LEAF_MASK_OVERRIDE),
            "n_predictions": len(df),
            "skipped_images": len(dataset._skipped),
        }
        print(f"  predictions rows: {len(df)} | skipped: {len(dataset._skipped)}")

        del model, loader, dataset
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    combined = _merge_per_model_frames(per_model_dfs)
    combined["mapped_source_class"] = combined["folder"].map(lambda f: folder_map.get(f) if f else None)

    model_names = [entry.name for entry in spec.checkpoints]
    for name in model_names:
        col = f"{name}_correct"
        combined[col] = (
            combined[f"{name}_top1_class"] == combined["mapped_source_class"]
        ) & combined["mapped_source_class"].notna()

    if model_names:
        preds_cols = [f"{n}_top1_class" for n in model_names]
        combined["all_models_agree"] = combined[preds_cols].apply(
            lambda row: len(set(row.dropna())) == 1,
            axis=1,
        )

    agreement = _agreement_matrix(combined, model_names)
    agreement_csv = dataset_out / "cross_model_agreement.csv"
    agreement.to_csv(agreement_csv)

    conf_plot = _write_confidence_plot(per_model_dfs, dataset_out)
    dis_plot, combined = _write_entropy_plot(combined, model_names, dataset_out)

    combined_csv = dataset_out / "all_models_predictions.csv"
    combined.to_csv(combined_csv, index=False)

    meta_json = dataset_out / "model_meta.json"
    meta_json.write_text(json.dumps(model_meta, indent=2, default=str))

    preflight_json = dataset_out / "dataset_preflight.json"
    preflight_json.write_text(json.dumps(asdict(spec), indent=2, default=str))

    summary_md = _write_run_summary(
        spec,
        dataset_out,
        combined,
        model_meta,
        str(agreement_csv),
        conf_plot,
        dis_plot,
    )

    mapped_mask = combined["mapped_source_class"].notna()
    return {
        "run_name": spec.run_name,
        "dataset_kind": spec.dataset_kind,
        "dataset_root": spec.dataset_root,
        "resolved_root": spec.inspection.resolved_root,
        "output_dir": str(dataset_out),
        "combined_csv": str(combined_csv),
        "agreement_csv": str(agreement_csv),
        "summary_md": summary_md,
        "model_meta_json": str(meta_json),
        "dataset_preflight_json": str(preflight_json),
        "confidence_plot": conf_plot,
        "disagreement_plot": dis_plot,
        "num_images": int(len(combined)),
        "mapped_images": int(mapped_mask.sum()),
        "unmapped_images": int((~mapped_mask).sum()),
        "all_models_agree_rate": float(combined["all_models_agree"].mean()) if "all_models_agree" in combined else float("nan"),
    }


In [ ]:
RUN_RESULTS = []
for spec in RUN_SPECS:
    RUN_RESULTS.append(_process_dataset_run(spec))

MASTER_SUMMARY_DF = pd.DataFrame(RUN_RESULTS).sort_values(["dataset_kind", "run_name"]).reset_index(drop=True)
MASTER_SUMMARY_CSV = Path(OUTPUT_DIR) / "dataset_run_summary.csv"
MASTER_SUMMARY_DF.to_csv(MASTER_SUMMARY_CSV, index=False)

PREFLIGHT_JSON = Path(OUTPUT_DIR) / "dataset_preflight_all.json"
PREFLIGHT_JSON.write_text(json.dumps([asdict(spec) for spec in RUN_SPECS], indent=2, default=str))

print("\nAll dataset runs processed.")
print(f"Master summary: {MASTER_SUMMARY_CSV}")
print(f"Preflight details: {PREFLIGHT_JSON}")
MASTER_SUMMARY_DF


## 7. Combined multi-dataset summary


In [ ]:
if MASTER_SUMMARY_DF.empty:
    raise RuntimeError("No dataset results were produced.")

summary_cols = [
    "run_name", "dataset_kind", "num_images", "mapped_images",
    "unmapped_images", "all_models_agree_rate", "output_dir",
]
MASTER_SUMMARY_DF[summary_cols]


## 8. Per-dataset artifact locations


In [ ]:
for row in RUN_RESULTS:
    print(f"\n=== {row['run_name']} ({row['dataset_kind']}) ===")
    print(f"Output dir:        {row['output_dir']}")
    print(f"Combined CSV:      {row['combined_csv']}")
    print(f"Agreement CSV:     {row['agreement_csv']}")
    print(f"Run summary:       {row['summary_md']}")
    print(f"Model meta JSON:   {row['model_meta_json']}")
    print(f"Preflight JSON:    {row['dataset_preflight_json']}")
    print(f"Confidence plot:   {row['confidence_plot']}")
    print(f"Disagreement plot: {row['disagreement_plot']}")


## 9. Optional: inspect one dataset run in detail


In [ ]:
FOCUS_RUN = RUN_RESULTS[-1]["run_name"]
focus = next(row for row in RUN_RESULTS if row["run_name"] == FOCUS_RUN)
focus_df = pd.read_csv(focus["combined_csv"])
print(f"Focused run: {FOCUS_RUN}")
print(f"Rows: {len(focus_df)}")
focus_df.head()


In [ ]:
focus = next(row for row in RUN_RESULTS if row["run_name"] == FOCUS_RUN)
agreement_df = pd.read_csv(focus["agreement_csv"], index_col=0)
print(f"Cross-model agreement for {FOCUS_RUN}:")
agreement_df


## 10. Summary & next steps


In [ ]:
print("All outputs in:", OUTPUT_DIR)
print(f"Master summary CSV: {MASTER_SUMMARY_CSV}")
print(f"Datasets processed: {len(RUN_RESULTS)}")
print("\nRecommended next steps:")
print("1. Start with dataset_run_summary.csv to compare agreement and coverage across datasets.")
print("2. Open each run's RUN_SUMMARY.md for per-dataset conclusions and artifact links.")
print("3. Inspect focused all_models_predictions.csv files for high-entropy or disagreement-heavy images.")
